# Lab #28 — Full Platform Integration Sprint

**AICB-P2T2 · Ngày 28 · Chương 6: Tổng Hợp**

Notebook này hướng dẫn từng bước ghép toàn bộ AI platform từ các lab trước (N16–N27) thành một hệ thống end-to-end.

## Mục tiêu

- Dựng infrastructure local bằng Docker Compose
- Kết nối GPU serving trên Kaggle qua tunnel (ngrok/cloudflared)
- Thực hiện 10 integration points: Kafka → Prefect → Delta Lake → Feast → Qdrant → API Gateway → vLLM
- Kiểm tra observability (Prometheus, Grafana, LangSmith)
- Chạy smoke tests và production readiness check

## Kiến trúc Hybrid

```
┌─────────────────────────────────────────────────────────┐
│                  LOCAL (Docker Compose)                  │
│  Kafka ──► Prefect ──► Delta Lake ──► Feast (Redis)     │
│     │                                    │               │
│     └──► Vector Store (Qdrant)           │               │
│                                          ▼               │
│  Prometheus ◄── Grafana          API Gateway (FastAPI) │
└──────────────────────────────────────────┼───────────────┘
                                           │  HTTP (tunnel)
┌──────────────────────────────────────────┼───────────────┐
│                 KAGGLE (GPU T4/P100)      │               │
│  vLLM serving ◄──────────────────────────┘               │
│  Embedding model (sentence-transformers)                 │
│  MLflow experiment tracking                              │
└─────────────────────────────────────────────────────────┘
```

## Yêu cầu trước khi bắt đầu

| Thành phần | Mô tả |
|------------|-------|
| Docker Desktop | Đang chạy trên máy local |
| Python 3.10+ | Để chạy scripts và notebook này |
| Kaggle GPU | Tài khoản Kaggle với GPU T4 x2 đã bật |
| Tunnel | `ngrok` hoặc `cloudflared` để expose Kaggle services |

**Thư mục làm việc:** Mở terminal và `cd` vào thư mục `lab28/` của repo này.

---
## PHẦN 0 — Cài đặt dependencies cho notebook local

Cell dưới cài các thư viện Python cần thiết để chạy các bước integration trên máy local.
Chỉ cần chạy **một lần** trước khi bắt đầu lab.

In [1]:
# Cài dependencies cho phần local (chạy trong thư mục lab28/)
!pip install -q kafka-python redis pandas pyarrow qdrant-client requests python-dotenv pytest mlflow

In [2]:
import os
from pathlib import Path

# Đảm bảo working directory là lab28/
LAB28_ROOT = Path.cwd()
if LAB28_ROOT.name != "lab28":
    candidate = LAB28_ROOT / "lab28"
    if candidate.exists():
        os.chdir(candidate)
        LAB28_ROOT = candidate

print(f"Working directory: {LAB28_ROOT}")
print(f"Files: {list(LAB28_ROOT.glob('*'))[:8]}...")

Working directory: /Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28
Files: [PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/LAB28_GUIDE.md'), PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/feast'), PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/api-gateway'), PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/delta-lake'), PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/smoke-tests'), PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/README.md'), PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/.gitignore'), PosixPath('/Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/.env')]...


---
## PHẦN 1 — Dựng Infrastructure Local (Docker Compose)

**Mục tiêu:** Khởi động Kafka, Prefect, Qdrant, Redis, Prometheus, Grafana và API Gateway trên máy local.

### Bước 1.1 — Cấu trúc thư mục

Repo đã có sẵn cấu trúc:

```
lab28/
├── docker-compose.yml      # Định nghĩa tất cả services
├── monitoring/prometheus.yml
├── prefect/flows/          # Prefect pipeline
├── api-gateway/            # FastAPI gateway
├── scripts/                # Integration scripts
├── smoke-tests/            # End-to-end tests
└── delta-lake/             # Lưu parquet (giả lập Delta Lake)
```

### Bước 1.2 — Các services trong Docker Compose

| Service | Port | Vai trò |
|---------|------|--------|
| Kafka + Zookeeper | 9092 | Message broker — nhận dữ liệu streaming |
| Prefect Orion | 4200 | Orchestration — điều phối pipeline |
| Qdrant | 6333 | Vector database — lưu embeddings |
| Redis | 6379 | Online feature store (Feast) |
| Prometheus | 9090 | Thu thập metrics |
| Grafana | 3000 | Dashboard visualization |
| API Gateway | 8000 | Entry point cho inference requests |

In [3]:
# Bước 1.3 — Khởi động Docker Compose stack
# Lệnh này pull images và start tất cả containers ở background (-d)
!docker compose up -d

WARN[0000] /Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
[+] up 9/9
 ✔ Container lab28-prometheus-1     Running                                 0.0s
 ✔ Container lab28-redis-1          Running                                 0.0s
 ✔ Container lab28-zookeeper-1      Running                                 0.0s
 ✔ Container lab28-qdrant-1         Running                                 0.0s
 ✔ Container lab28-prefect-orion-1  Running                                 0.0s
 ✔ Container lab28-api-gateway-1    Running                                 0.0s
 ✔ Container lab28-kafka-1          Running                                 0.0s
 ✔ Container lab28-prefect-worker-1 Running                                 0.0s
 ✔ Container lab28-grafana-1        Running                                 0.0s


In [4]:
# Bước 1.4 — Kiểm tra trạng thái các services
!docker compose ps

WARN[0000] /Users/harrietlesly/Documents/VinUni-Day28-Track02/lab28/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion 
NAME                     IMAGE                                 COMMAND                  SERVICE          CREATED              STATUS              PORTS
lab28-api-gateway-1      lab28-api-gateway                     "uvicorn main:app --…"   api-gateway      About a minute ago   Up About a minute   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
lab28-grafana-1          grafana/grafana:latest                "/run.sh"                grafana          4 minutes ago        Up 4 minutes        0.0.0.0:3000->3000/tcp, [::]:3000->3000/tcp
lab28-kafka-1            confluentinc/cp-kafka:7.5.0           "/etc/confluent/dock…"   kafka            4 minutes ago        Up 4 minutes        0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp
lab28-prefect-orion-1    prefecthq/prefect:2.14.0-python3.10   "/usr/bin/tini -g --…" 

In [5]:
import requests

# Kiểm tra nhanh các endpoint quan trọng
checks = {
    "Prefect UI": "http://localhost:4200/api/health",
    "Qdrant": "http://localhost:6333/healthz",
    "Prometheus": "http://localhost:9090/-/healthy",
    "Grafana": "http://localhost:3000/api/health",
    "API Gateway": "http://localhost:8000/health",
}

for name, url in checks.items():
    try:
        r = requests.get(url, timeout=5)
        status = "✅ OK" if r.status_code == 200 else f"⚠️ HTTP {r.status_code}"
    except Exception as e:
        status = f"❌ {e}"
    print(f"{name:20s} → {status}")

Prefect UI           → ✅ OK
Qdrant               → ✅ OK
Prometheus           → ✅ OK
Grafana              → ✅ OK
API Gateway          → ✅ OK


> **Lưu ý:** Nếu API Gateway chưa healthy, có thể do chưa cấu hình `VLLM_NGROK_URL` trong file `.env`. Hoàn thành Phần 2 trước, sau đó restart: `docker compose up -d api-gateway`

---
## PHẦN 2 — Kaggle GPU Setup & Tunnel

**Mục tiêu:** Chạy vLLM (LLM serving) và Embedding service trên Kaggle GPU, expose ra internet để local gọi được.

> ⚠️ **Các cell dưới chạy trên Kaggle Notebook** (bật GPU T4 x2), không phải trên máy local.

### Tại sao tách GPU lên Kaggle?

- **Cost efficiency:** GPU trên cloud miễn phí/cheap hơn mua hardware
- **Local nhẹ hơn:** Máy local chỉ chạy orchestration và data pipeline
- **Separation of concerns:** Inference layer tách biệt khỏi data layer

### Bước 2.1 — Cài dependencies trên Kaggle

In [ ]:
# [KAGGLE] Cell 1 — Cài dependencies
!pip install -q vllm fastapi uvicorn pyngrok mlflow sentence-transformers

# Nếu cài vLLM bị lỗi, dùng fallback:
# !pip install transformers==4.46.3 --quiet
# !pip install vllm==0.7.3 --quiet

### Bước 2.2 — Cấu hình ngrok token

Đăng ký tại [ngrok.com](https://ngrok.com) và lấy authtoken tại trang dashboard.

In [ ]:
# [KAGGLE] Cell 2 — Setup ngrok
from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # ← Thay bằng token của bạn

### Bước 2.3 — Khởi động vLLM server (Single GPU)

Model `Qwen2.5-7B-Instruct-GPTQ-Int4` được quantize 4-bit để fit vào GPU T4.
Server chạy background thread, mất ~60 giây để load model.

In [ ]:
# [KAGGLE] Cell 3 — Khởi động vLLM server
import subprocess
import threading
import time

def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.5"
    ])

thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()
time.sleep(60)  # Chờ model load vào GPU
print("vLLM server started trên port 8001")

In [ ]:
# [KAGGLE] Cell 4 — Tạo ngrok tunnel cho vLLM
tunnel = ngrok.connect(8001, "http")
vllm_url = tunnel.public_url
print(f"vLLM URL (copy vào .env): {vllm_url}")

### Bước 2.4 — Embedding service trên Kaggle

Model `BAAI/bge-small-en-v1.5` tạo vector 384 chiều — khớp với Qdrant collection config.

In [ ]:
# [KAGGLE] Cell 5 — Embedding API server
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn
import threading

app = FastAPI()
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = model.encode(texts).tolist()
    return {"embeddings": embeddings}

def run_embed():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run_embed, daemon=True).start()
embed_tunnel = ngrok.connect(8002, "http")
embed_url = embed_tunnel.public_url
print(f"Embedding URL (copy vào .env): {embed_url}")

### Bước 2.5 — MLflow tracking (Integration 6 & 7)

Ghi lại metadata của model serving vào **MLflow Model Registry** — theo dõi model nào đang serve, URL, hyperparameters.

| Cách | Khi nào dùng | Tracking URI |
|------|--------------|--------------|
| **Local** (khuyến nghị cho lab) | Chạy notebook trên máy, không cần DagsHub | `http://localhost:5000` hoặc `file:./mlruns` |
| **Kaggle + DagsHub** | Deploy thật, team collaboration | `https://dagshub.com/YOUR_USER/lab28.mlflow` |
| **Kaggle only** | Log ngay trên notebook GPU | DagsHub hoặc file store trên Kaggle |

> **Local:** Xem **Bước 2.5b** bên dưới — khởi động MLflow server và log experiment ngay trong notebook này.
>
> **Kaggle:** Cell 6 — chạy trên Kaggle sau khi có `vllm_url`.

In [ ]:
# [KAGGLE] Cell 6 — MLflow trên Kaggle (DagsHub hoặc file store)
# Chỉ chạy trên Kaggle Notebook sau khi Cell 4 đã có biến vllm_url
import mlflow

# Option A: DagsHub (remote, cần tài khoản + token)
# mlflow.set_tracking_uri("https://dagshub.com/YOUR_USER/lab28.mlflow")

# Option B: File store trên Kaggle (đơn giản, không cần server)
mlflow.set_tracking_uri("file:./mlruns")

mlflow.set_experiment("lab28-integration")

with mlflow.start_run(run_name="vllm-serving-v1-kaggle"):
    mlflow.log_param("model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4")
    mlflow.log_param("max_model_len", 4096)
    mlflow.log_metric("gpu_memory_utilization", 0.85)
    mlflow.log_metric("avg_latency_ms", 450)
    mlflow.set_tag("serving_url", vllm_url)
    mlflow.set_tag("status", "production")
    mlflow.set_tag("environment", "kaggle-gpu")

print("Integration 6+7 OK (Kaggle): MLflow → Model Registry → vLLM")

### Bước 2.5b — MLflow Local (chạy trên máy)

**3 bước:**
1. Khởi động MLflow Tracking Server (UI tại http://localhost:5000)
2. Log experiment với metadata model vLLM
3. Mở UI xem runs, params, metrics, tags

**Không cần Kaggle GPU** — dùng URL từ `.env` (`VLLM_NGROK_URL`) hoặc placeholder nếu chưa setup tunnel.

```
mlflow/
├── mlflow.db          ← SQLite backend (experiments, runs)
└── artifacts/         ← Model artifacts (nếu có)
```

In [ ]:
# [LOCAL] Bước 1 — Khởi động MLflow Tracking Server
import subprocess
import time
from pathlib import Path

import requests

MLFLOW_DIR = Path("mlflow")
MLFLOW_DIR.mkdir(exist_ok=True)
(MLFLOW_DIR / "artifacts").mkdir(exist_ok=True)

DB_PATH = MLFLOW_DIR / "mlflow.db"
MLFLOW_URI = "http://localhost:5000"

def is_mlflow_up():
    try:
        return requests.get(f"{MLFLOW_URI}/health", timeout=2).status_code == 200
    except Exception:
        return False

if is_mlflow_up():
    print(f"MLflow server đã chạy → {MLFLOW_URI}")
else:
    subprocess.Popen(
        [
            "mlflow", "server",
            "--backend-store-uri", f"sqlite:///{DB_PATH.resolve()}",
            "--default-artifact-root", str((MLFLOW_DIR / "artifacts").resolve()),
            "--host", "127.0.0.1",
            "--port", "5000",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(15):
        if is_mlflow_up():
            break
        time.sleep(1)
    print(f"MLflow server started → {MLFLOW_URI}" if is_mlflow_up() else "⚠️ Không start được MLflow — thử: mlflow server --port 5000")

In [ ]:
# [LOCAL] Bước 2 — Log experiment vào MLflow Model Registry
import os

import mlflow
from dotenv import load_dotenv

load_dotenv()

# URL vLLM: lấy từ .env sau khi setup Kaggle tunnel, hoặc placeholder để test
vllm_url_local = os.getenv("VLLM_NGROK_URL", "http://localhost:8001-not-configured")

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("lab28-integration")

with mlflow.start_run(run_name="vllm-serving-v1-local") as run:
    mlflow.log_param("model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4")
    mlflow.log_param("max_model_len", 4096)
    mlflow.log_param("serving_backend", "vllm")
    mlflow.log_metric("gpu_memory_utilization", 0.85)
    mlflow.log_metric("avg_latency_ms", 450)
    mlflow.set_tag("serving_url", vllm_url_local)
    mlflow.set_tag("status", "production")
    mlflow.set_tag("environment", "local-hybrid")
    run_id = run.info.run_id

print(f"Integration 6+7 OK (Local): run_id = {run_id}")
print(f"Xem UI: {MLFLOW_URI}/#/experiments")
print(f"Serving URL logged: {vllm_url_local}")

In [ ]:
# [LOCAL] Bước 3 — Xác minh runs đã ghi (API + tóm tắt)
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=MLFLOW_URI)
experiment = client.get_experiment_by_name("lab28-integration")
runs = client.search_runs(experiment_ids=[experiment.experiment_id], max_results=5)

print(f"Experiment: {experiment.name} (id={experiment.experiment_id})")
print(f"Số runs: {len(runs)}\n")

for r in runs:
    print(f"  • {r.info.run_name}")
    print(f"    status={r.data.tags.get('status')}  env={r.data.tags.get('environment')}")
    print(f"    serving_url={r.data.tags.get('serving_url')}")
    print(f"    model={r.data.params.get('model')}\n")

print(f"👉 Mở trình duyệt: {MLFLOW_URI}")

### Bước 2.6 — Cập nhật `.env` trên local

Sau khi có URLs từ Kaggle, cập nhật file `lab28/.env`:

In [ ]:
# [LOCAL] Tạo hoặc cập nhật file .env
from dotenv import load_dotenv

env_content = """\
VLLM_NGROK_URL=https://xxxx.ngrok-free.app
EMBED_NGROK_URL=https://yyyy.ngrok-free.app
LANGCHAIN_API_KEY=your_langsmith_key
LANGCHAIN_PROJECT=lab28-platform
"""

env_path = Path(".env")
if not env_path.exists():
    env_path.write_text(env_content)
    print("Đã tạo file .env — hãy thay URLs và API key thật!")
else:
    print("File .env đã tồn tại. Kiểm tra nội dung:")

load_dotenv()
print(f"VLLM_NGROK_URL  = {os.getenv('VLLM_NGROK_URL', '(chưa set)')}")
print(f"EMBED_NGROK_URL = {os.getenv('EMBED_NGROK_URL', '(chưa set)')}")

In [ ]:
# Restart API Gateway sau khi cập nhật .env
!docker compose up -d api-gateway

---
## PHẦN 3 — 10 Integration Points

Đây là phần cốt lõi: kết nối từng thành phần trong pipeline data → inference.

```
Data Ingestion → Kafka → Prefect → Delta Lake → Feast (Redis)
                                              ↓
                                    Embeddings → Qdrant
                                              ↓
                              API Gateway ← vLLM (Kaggle)
                                              ↓
                              Prometheus / Grafana / LangSmith
```

### Integration 1: Data Ingestion → Kafka

**Giải thích:** Gửi dữ liệu thô vào Kafka topic `data.raw`. Kafka đóng vai trò message broker — decouple producer (ingestion) khỏi consumer (Prefect pipeline).

**Tại sao dùng Kafka?**
- **Decoupling:** Producer và consumer không cần biết nhau
- **Replay:** Có thể đọc lại messages từ offset cũ
- **Scalability:** Nhiều consumer có thể đọc cùng topic

In [ ]:
# Integration 1 — Ingest data vào Kafka
# (Tương đương scripts/01_ingest_to_kafka.py)
from kafka import KafkaProducer
import json
import time

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode()
)

def ingest_data(records: list[dict]):
    for record in records:
        producer.send("data.raw", value=record)
        print(f"Sent: {record['id']}")
    producer.flush()

sample_data = [
    {"id": "doc_001", "text": "AI platform integration test", "timestamp": time.time()},
    {"id": "doc_002", "text": "Kafka to Prefect pipeline", "timestamp": time.time()},
    {"id": "doc_003", "text": "Vector search with Qdrant", "timestamp": time.time()},
]

ingest_data(sample_data)
print("\n✅ Integration 1 OK: Data → Kafka")

### Integration 2: Kafka → Prefect Pipeline

**Giải thích:** Prefect flow `kafka_to_delta_flow` consume messages từ Kafka topic `data.raw`, xử lý và lưu vào Delta Lake (dạng parquet).

**Flow gồm 2 tasks:**
1. `consume_and_process()` — đọc messages từ Kafka
2. `save_to_delta()` — ghi DataFrame ra file parquet

> File: `prefect/flows/kafka_to_delta.py` — deploy bằng lệnh bên dưới hoặc chạy flow trực tiếp để test.

In [ ]:
# Integration 2 — Chạy Prefect flow thủ công (test local, không qua worker)
import sys
sys.path.insert(0, "prefect/flows")

from kafka import KafkaConsumer
import pandas as pd
from datetime import datetime

def consume_kafka_local():
    """Simulate Prefect task: consume từ Kafka trên localhost"""
    consumer = KafkaConsumer(
        "data.raw",
        bootstrap_servers="localhost:9092",
        auto_offset_reset="earliest",
        consumer_timeout_ms=5000,
        value_deserializer=lambda m: json.loads(m.decode())
    )
    records = [msg.value for msg in consumer]
    print(f"Consumed {len(records)} records from Kafka")
    return records

def save_to_delta_local(records):
    """Simulate Prefect task: lưu parquet vào delta-lake/raw/"""
    if not records:
        print("No records to save")
        return
    df = pd.DataFrame(records)
    path = Path("delta-lake/raw")
    path.mkdir(parents=True, exist_ok=True)
    outfile = path / f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}.parquet"
    df.to_parquet(outfile)
    print(f"Saved {len(df)} records → {outfile}")

records = consume_kafka_local()
save_to_delta_local(records)
print("\n✅ Integration 2 OK: Kafka → Delta Lake (parquet)")

In [ ]:
# Deploy flow lên Prefect Orion (chạy khi cần production schedule)
!cd prefect/flows && pip install -q -r requirements.txt && python kafka_to_delta.py

### Integration 3 & 4: Delta Lake → Feature Store (Feast/Redis)

**Giải thích:** Đọc parquet files từ Delta Lake, push features vào Redis (online store của Feast).

**Feast pattern:**
- **Offline store:** Delta Lake / parquet (historical data)
- **Online store:** Redis (low-latency lookup cho inference)

Key format: `feature:{doc_id}` → JSON với text, timestamp, processed flag.

In [ ]:
# Integration 3+4 — Delta Lake → Feast (Redis)
# (Tương đương scripts/03_delta_to_feast.py)
import glob
import redis

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

files = glob.glob("delta-lake/raw/*.parquet")
if not files:
    print("⚠️ Chưa có data trong Delta Lake — chạy Integration 2 trước")
else:
    df = pd.concat([pd.read_parquet(f) for f in files])
    print(f"Loaded {len(df)} records from Delta Lake")

    for _, row in df.iterrows():
        feature_key = f"feature:{row['id']}"
        r.set(feature_key, json.dumps({
            "text": row["text"],
            "timestamp": row["timestamp"],
            "processed": True
        }))

    keys = r.keys("feature:*")
    print(f"\n✅ Integration 3+4 OK: {len(keys)} features trong Feast (Redis)")

### Integration 5: Data → Vector Store (Embeddings → Qdrant)

**Giải thích:**
1. Gọi Embedding API trên Kaggle để chuyển text thành vector 384 chiều
2. Lưu vectors vào Qdrant collection `documents`

**Qdrant** dùng cosine similarity để tìm documents liên quan nhất khi inference (RAG pattern).

In [ ]:
# Integration 5 — Embed và lưu vào Qdrant
# (Tương đương scripts/05_embed_to_qdrant.py)
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

EMBED_URL = os.environ.get("EMBED_NGROK_URL")
if not EMBED_URL:
    print("⚠️ EMBED_NGROK_URL chưa set — bỏ qua hoặc set trong .env")
else:
    qdrant = QdrantClient(host="localhost", port=6333)

    qdrant.recreate_collection(
        collection_name="documents",
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

    records = [
        {"id": "doc_001", "text": "AI platform integration test"},
        {"id": "doc_002", "text": "Kafka to Prefect pipeline"},
    ]

    response = requests.post(
        f"{EMBED_URL}/embed",
        json={"texts": [r["text"] for r in records]},
        timeout=30
    )
    embeddings = response.json()["embeddings"]

    points = [
        PointStruct(id=i, vector=emb, payload=rec)
        for i, (emb, rec) in enumerate(zip(embeddings, records))
    ]
    qdrant.upsert(collection_name="documents", points=points)
    print(f"\n✅ Integration 5 OK: {len(points)} vectors stored in Qdrant")

### Integration 6 & 7: MLflow → Model Registry → vLLM

**Đã thực hiện ở Phần 2:**
- **Local:** Bước 2.5b — MLflow server tại http://localhost:5000
- **Kaggle:** Cell 6 — log trực tiếp trên GPU notebook

MLflow track:
- Model name, hyperparameters (`max_model_len`, `gpu_memory_utilization`)
- Serving URL (ngrok tunnel tới vLLM)
- Tag `status=production` — đánh dấu model đang serve

### Integration 8: API Gateway — Serving → Inference

**Giải thích flow `/api/v1/chat`:**

1. Nhận `query` + `embedding` từ client
2. **Vector search** trên Qdrant — tìm 3 documents liên quan nhất
3. **LLM inference** — gửi prompt (context + query) tới vLLM trên Kaggle
4. Trả về `answer`, `latency_ms`, `model`

File: `api-gateway/main.py`

In [ ]:
# Integration 8 — Test API Gateway chat endpoint
import numpy as np

payload = {
    "query": "What is platform engineering?",
    "embedding": [0.1] * 384  # Trong production: dùng embedding thật từ Kaggle
}

try:
    resp = requests.post(
        "http://localhost:8000/api/v1/chat",
        json=payload,
        timeout=60
    )
    if resp.status_code == 200:
        data = resp.json()
        print(f"Answer: {data['answer'][:200]}...")
        print(f"Latency: {data['latency_ms']} ms")
        print(f"Model: {data['model']}")
        print("\n✅ Integration 8 OK: API Gateway → vLLM")
    else:
        print(f"⚠️ HTTP {resp.status_code}: {resp.text[:300]}")
except Exception as e:
    print(f"❌ Lỗi: {e}")
    print("Kiểm tra: docker compose logs api-gateway")

### Integration 9 & 10: Observability (Prometheus + LangSmith)

**Integration 9 — Prometheus:** API Gateway expose `/metrics` endpoint. Prometheus scrape mỗi 15 giây.

**Integration 10 — LangSmith:** Trace LLM calls qua LangChain SDK (cần `LANGCHAIN_API_KEY`).

In [ ]:
# Integration 9 — Kiểm tra Prometheus metrics
def check_prometheus():
    resp = requests.get(
        "http://localhost:9090/api/v1/query",
        params={"query": 'up{job="api-gateway"}'}
    )
    data = resp.json()
    assert data["status"] == "success"
    results = data["data"]["result"]
    if results:
        print(f"API Gateway up = {results[0]['value'][1]}")
    print("✅ Integration 9 OK: Prometheus metrics flowing")

check_prometheus()

# Kiểm tra metrics endpoint trực tiếp
metrics = requests.get("http://localhost:8000/metrics", timeout=5)
print(f"Metrics endpoint: HTTP {metrics.status_code}")
print(f"Sample lines: {metrics.text[:200]}...")

In [ ]:
# Integration 10 — Kiểm tra LangSmith (optional, cần API key)
langchain_key = os.environ.get("LANGCHAIN_API_KEY")

if not langchain_key or langchain_key == "your_langsmith_key":
    print("⚠️ LANGCHAIN_API_KEY chưa set — bỏ qua LangSmith check")
else:
    from langsmith import Client
    client = Client(api_key=langchain_key)
    runs = list(client.list_runs(project_name="lab28-platform", limit=1))
    if runs:
        print(f"Latest run: {runs[0].name}")
        print("✅ Integration 10 OK: LangSmith traces visible")
    else:
        print("Chưa có traces — gửi vài request chat trước")

---
## PHẦN 4 — Smoke Tests (End-to-End)

5 test classes kiểm tra toàn bộ user journeys:

| Test | Journey |
|------|--------|
| TestHappyPath | Full inference request qua API Gateway |
| TestDataIngestion | Kafka ingest → Qdrant store |
| TestObservability | Prometheus scrape + Grafana health |
| TestFailurePath | Invalid request + timeout handling |
| TestFeatureStore | Feast (Redis) có features |

In [ ]:
# Chạy smoke tests
!pytest smoke-tests/ -v --tb=short

---
## PHẦN 5 — Production Readiness Check

Script tự động kiểm tra 10+ tiêu chí production:
- **Reliability:** health check, API docs
- **Observability:** Prometheus, Grafana, metrics endpoint
- **Security:** unauthorized access rejected
- **Data stores:** Qdrant, Redis, Kafka topics

**Target:** Score ≥ 80%

In [ ]:
# Production readiness check
!python scripts/production_readiness_check.py

---
## PHẦN 6 — Demo Quick Reference

### URLs quan trọng

| Service | URL | Credentials |
|---------|-----|-------------|
| Prefect UI | http://localhost:4200 | — |
| Grafana | http://localhost:3000 | admin / admin |
| Qdrant Dashboard | http://localhost:6333/dashboard | — |
| Prometheus | http://localhost:9090 | — |
| MLflow UI | http://localhost:5000 | — |
| API Gateway | http://localhost:8000/docs | — |

### Demo flow (15 phút)

1. **Architecture overview** (2 phút) — giải thích 5 layers và hybrid pattern
2. **Happy path** (5 phút) — ingest → pipeline → chat API
3. **Error scenario** (3 phút) — timeout, stop Qdrant, graceful degradation
4. **Observability** (3 phút) — Grafana dashboards, load test
5. **Q&A** (2 phút)

In [ ]:
# Load test nhỏ — tạo data cho Grafana graphs
import concurrent.futures

def send_request(i):
    try:
        requests.post(
            "http://localhost:8000/api/v1/chat",
            json={"query": f"load test {i}", "embedding": [0.1] * 384},
            timeout=30
        )
    except Exception:
        pass

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    list(executor.map(send_request, range(1, 11)))

print("Đã gửi 10 requests — kiểm tra Grafana tại http://localhost:3000")

---
## Checklist cuối buổi

```
[ ] docker compose ps — tất cả services Up
[ ] Integration 1: Data → Kafka OK
[ ] Integration 2: Prefect flow deploy thành công
[ ] Integration 3+4: Delta Lake → Feast (Redis) OK
[ ] Integration 5: Embeddings → Qdrant OK
[ ] Integration 8: curl /api/v1/chat — nhận response
[ ] pytest smoke-tests/ -v — 5/5 PASSED
[ ] production_readiness_check.py — score >= 80%
[ ] Grafana dashboard có metrics
[ ] Kaggle notebook vẫn chạy (kernel active)
[ ] Demo script đã chạy thử 1 lần
```

---

## Timeline 2 giờ

| Thời gian | Công việc |
|-----------|----------|
| 0:00 – 0:20 | Phần 1: Docker Compose up |
| 0:20 – 0:35 | Phần 2: Kaggle notebook, lấy tunnel URLs |
| 0:35 – 1:00 | Phần 3: 10 integration points |
| 1:00 – 1:20 | Phần 4: Smoke tests |
| 1:20 – 1:35 | Phần 5: Production readiness |
| 1:35 – 2:00 | Phần 6: Rehearse demo |

---
## Troubleshooting

**Services không start:**
```bash
docker compose logs <service_name>
docker compose down -v && docker compose up -d
```

**Prefect worker không connect:**
- Kiểm tra Prefect UI: http://localhost:4200
- `docker compose logs prefect-worker`

**API Gateway lỗi vLLM:**
- Kiểm tra Kaggle kernel còn active
- ngrok tunnel có thể expire — tạo lại và update `.env`

**Kafka consumer lag:**
```bash
docker exec lab28-kafka-1 kafka-topics --list --bootstrap-server localhost:9092
```